# Step-by-Step A2A (Agent-to-Agent) Experiment

This notebook walks you through building a multi-agent system using the **A2A protocol** — the same way you explored RAG and MCP step-by-step.

---

## What is A2A?

**A2A (Agent-to-Agent)** is an open protocol (created by Google, adopted by many AI vendors) that defines how AI agents communicate with **each other** over HTTP.

### The gap that A2A fills

You already know MCP: an LLM uses **tools** (vertical relationship — model → tools).
A2A solves a different problem: agents **delegating tasks** to other agents (horizontal relationship — agent ↔ agent).

```
  MCP (Model ↔ Tools)
  ─────────────────────────────────────────
  LLM  ──────────────►  calculator_server
        tool call               │
        ◄──────────────  42.0  ─┘

  A2A (Agent ↔ Agent)
  ─────────────────────────────────────────
  User
    │
    ▼
  Orchestrator Agent   ──► Math Agent
  (decides who to ask)  ◄── "The answer is 42"
                        ──► Writer Agent
                        ◄── "Here is your essay..."
```

### Key concepts

| Concept | What it is | Example |
|---|---|---|
| **Agent Card** | JSON descriptor advertising an agent's capabilities | `/.well-known/agent.json` |
| **Skill** | A specific capability an agent advertises | `"math"`, `"summarize"` |
| **Task** | The unit of work sent to an agent | A user question + conversation history |
| **Message** | A turn in the conversation (user or agent) | Text, file, or structured data |
| **Part** | A piece of a message | `TextPart`, `FilePart`, `DataPart` |
| **Artifact** | The agent's output | The final answer or generated file |

### A2A Transport

Unlike MCP (which defaults to stdio), A2A uses **HTTP + JSON-RPC 2.0**.
Agents are real HTTP services — discoverable, network-addressable, and composable.

```
  Client                         Agent Server
  ──────────────────────────────────────────────
  GET /.well-known/agent.json  ──►  agent card
  POST /  {method: "message/send", ...}  ──►  execute
                               ◄──  Task or Message
```

## Step 1 — Install dependencies

The Python SDK for A2A is `a2a-sdk`. We also need:
- `uvicorn`: ASGI server to run agent HTTP services
- `httpx`: async HTTP client for the A2A client side

> Note: `ollama` is already installed from `requirements.txt`.

In [ ]:
%pip install a2a-sdk uvicorn httpx nest_asyncio2 --quiet

## Step 2 — The Agent Card: how agents advertise themselves

An **Agent Card** is a JSON document served at `GET /.well-known/agent.json`.
It is the equivalent of an OpenAPI spec for agents — any A2A client can discover
what the agent does and how to talk to it.

Let's build one in Python to see its structure:

In [1]:
import json
from a2a.types import AgentCard, AgentCapabilities, AgentSkill

sample_card = AgentCard(
    name="Math Agent",
    description="Solves math problems step by step.",
    url="http://localhost:10002/",        # where the agent is reachable
    version="1.0.0",
    capabilities=AgentCapabilities(
        streaming=False,                 # does it support SSE streaming?
        pushNotifications=False,         # can it push task updates?
    ),
    skills=[
        AgentSkill(
            id="math-solver",
            name="Math Solver",
            description="Solves arithmetic, algebra, and basic calculus.",
            tags=["math", "calculation"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

# This is exactly what a client sees at /.well-known/agent.json
print(json.dumps(sample_card.model_dump(exclude_none=True), indent=2))

{
  "capabilities": {
    "pushNotifications": false,
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Solves math problems step by step.",
  "name": "Math Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Solves arithmetic, algebra, and basic calculus.",
      "id": "math-solver",
      "name": "Math Solver",
      "tags": [
        "math",
        "calculation"
      ]
    }
  ],
  "url": "http://localhost:10002/",
  "version": "1.0.0"
}


## Step 3 — Write your first A2A agent: the Echo Agent

An A2A agent server has three parts:

1. **`AgentExecutor`** — your business logic (subclass this)
2. **`DefaultRequestHandler`** — wires executor to the A2A protocol
3. **`A2AStarletteApplication`** — the ASGI HTTP app (served via uvicorn)

```
  HTTP Request
      │
      ▼
  A2AStarletteApplication   (routing, JSON-RPC parsing)
      │
      ▼
  DefaultRequestHandler     (task lifecycle management)
      │
      ▼
  AgentExecutor.execute()   ← YOUR CODE GOES HERE
      │
      ▼
  EventQueue.enqueue_event()   (sends response back)
```

Let's write a minimal echo agent:

In [2]:
echo_agent_code = '''
# echo_agent.py — the simplest possible A2A agent

import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message


class EchoAgentExecutor(AgentExecutor):
    """Receives any text and echoes it back with a prefix."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input() extracts the text from the latest user message
        user_input = context.get_user_input()

        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(
            new_agent_text_message(f"Echo: {user_input}")
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Called if a client cancels the task mid-flight
        pass


# --- Agent Card -----------------------------------------------------------
agent_card = AgentCard(
    name="Echo Agent",
    description="Echoes back whatever you send it.",
    url="http://localhost:10001/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="echo",
            name="Echo",
            description="Echo the input message back.",
            tags=["echo", "test"]
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

# --- Wire everything together -------------------------------------------
request_handler = DefaultRequestHandler(
    agent_executor=EchoAgentExecutor(),
    task_store=InMemoryTaskStore(),  # stores task state in memory
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10001, log_level="warning")
'''

with open("echo_agent.py", "w") as f:
    f.write(echo_agent_code.strip())

print("echo_agent.py written!")

echo_agent.py written!


## Step 4 — Start the agent and inspect its Agent Card

Unlike MCP (where the client *spawns* the server), A2A agents run as **independent HTTP services**.
We start the agent as a background subprocess and then connect to it over HTTP.

In [3]:
import subprocess
import time
import httpx
import asyncio
import nest_asyncio

nest_asyncio.apply()  # allow asyncio in Jupyter

# Start the echo agent as a background process
echo_proc = subprocess.Popen(
    ["python3", "echo_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)  # give uvicorn a moment to start

# Verify it is running
if echo_proc.poll() is None:
    print("Echo agent is running (PID:", echo_proc.pid, ")")
else:
    stdout, stderr = echo_proc.communicate()
    print("Agent failed to start!")
    print(stderr.decode())

Echo agent is running (PID: 101126 )


In [4]:
import json

# Fetch the Agent Card — this is what any A2A client does first
# The well-known path is a convention from the A2A spec
async def fetch_agent_card(base_url: str):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{base_url}/.well-known/agent.json")
        return response.json()

card = asyncio.get_event_loop().run_until_complete(
    fetch_agent_card("http://localhost:10001")
)

print("=== Agent Card at /.well-known/agent.json ===")
print(json.dumps(card, indent=2))

=== Agent Card at /.well-known/agent.json ===
{
  "capabilities": {
    "streaming": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "Echoes back whatever you send it.",
  "name": "Echo Agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Echo the input message back.",
      "id": "echo",
      "name": "Echo",
      "tags": [
        "echo",
        "test"
      ]
    }
  ],
  "url": "http://localhost:10001/",
  "version": "1.0.0"
}


## Step 5 — Send your first A2A message

Now let's use `A2AClient` to send a message and receive a response.

The protocol uses **JSON-RPC 2.0** over HTTP POST to `/`.
The main method is `message/send`.

```
  POST http://localhost:10001/
  {
    "jsonrpc": "2.0",
    "id": "<request-id>",
    "method": "message/send",
    "params": {
      "message": {
        "role": "user",
        "parts": [{"kind": "text", "text": "hello!"}],
        "messageId": "<uuid>"
      }
    }
  }
```

In [8]:
# New API (a2a-sdk >= 0.3.x)
#
# What changed from 0.2.x:
#   OLD: A2AClient(httpx_client=..., agent_card=...)  → deprecated
#   NEW: ClientFactory.connect(url)                   → resolves card + creates client
#
#   OLD: client.send_message(SendMessageRequest(...)) → returns a response object
#   NEW: client.send_message(Message(...))            → async generator that yields events
#
# send_message now yields either:
#   - (Task, None)  — the completed task (non-streaming path)
#   - Message       — a direct message reply (rare)
#
# create_text_message_object(role, content='') — role is first, use content= keyword!

from a2a.client import ClientFactory
from a2a.client.helpers import create_text_message_object
from a2a.types import Task, Message


def extract_text(event) -> str:
    """Pull the agent's reply text from a send_message event."""
    if isinstance(event, tuple):
        # Non-streaming path: yields (Task, None)
        task, _ = event
        # Agent reply is the last 'agent' message in the task history
        for msg in reversed(task.history or []):
            if msg.role.value == "agent":
                for part in msg.parts:
                    if hasattr(part.root, "text"):
                        return part.root.text
    elif isinstance(event, Message):
        # Direct message reply path
        for part in event.parts:
            if hasattr(part.root, "text"):
                return part.root.text
    return str(event)


async def call_agent(base_url: str, text: str) -> str:
    # ClientFactory.connect() fetches the agent card and picks the right transport
    client = await ClientFactory.connect(base_url)
    # content= keyword required — first positional arg is role, not content
    async for event in client.send_message(create_text_message_object(content=text)):
        return extract_text(event)
    return ""


reply = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10001", "Hello, world!")
)
print(f"Agent replied: {reply}")

Agent replied: Echo: Hello, world!


In [9]:
# Inspect the raw event to understand what send_message yields
async def call_agent_raw(base_url: str, text: str):
    client = await ClientFactory.connect(base_url)
    async for event in client.send_message(create_text_message_object(content=text)):
        return event  # return the raw Python object

raw = asyncio.get_event_loop().run_until_complete(
    call_agent_raw("http://localhost:10001", "Show me the protocol!")
)

print(f"Type  : {type(raw)}")
print()

if isinstance(raw, tuple):
    task, _ = raw
    print(f"Task id     : {task.id}")
    print(f"Task status : {task.status.state}")
    print(f"History     :")
    for msg in task.history or []:
        print(f"  [{msg.role.value}] ", end="")
        for part in msg.parts:
            if hasattr(part.root, "text"):
                print(part.root.text[:120])
elif isinstance(raw, Message):
    print(f"Message role : {raw.role.value}")
    for part in raw.parts:
        if hasattr(part.root, "text"):
            print(part.root.text)

Type  : <class 'a2a.types.Message'>

Message role : agent
Echo: Show me the protocol!


## Step 6 — What just happened? The A2A task lifecycle

Reading the raw JSON above, you can see the full structure:

```
  response
  └── result  (a Task object)
      ├── id          — unique task identifier
      ├── status
      │   └── state   — "completed" | "working" | "failed" | "canceled"
      ├── history     — the conversation turns
      │   ├── {role: "user",  parts: [{kind: "text", text: "..."}]}
      │   └── {role: "agent", parts: [{kind: "text", text: "..."}]}
      └── artifacts   — structured outputs (files, data)
```

### Task states

```
  submitted ──► working ──► completed
                   │
                   ├──► failed
                   └──► canceled
```

The `DefaultRequestHandler` manages this lifecycle automatically.
Your `AgentExecutor.execute()` only needs to `enqueue_event()` with the result —
the framework handles state transitions, task storage, and error wrapping.

In [10]:
# Stop the echo agent — we are done with it
echo_proc.terminate()
echo_proc.wait()
print("Echo agent stopped.")

Echo agent stopped.


## Step 7 — Build a real agent backed by Ollama: the Math Agent

Now let's make an agent that actually *thinks* using a local LLM.
The pattern is simple:

```
  A2A Client
      │  message/send
      ▼
  MathAgentExecutor.execute()
      │  ollama.chat(model="llama3.2", messages=[...])
      ▼
  Ollama (llama3.2 running locally)
      │  "The answer is 42..."
      ▼
  event_queue.enqueue_event(new_agent_text_message(answer))
      │
      ▼
  A2A Client receives Task with completed status
```

> Make sure Ollama is running: `ollama serve`

In [ ]:
math_agent_code = '''
# math_agent.py — an A2A agent that solves math using llama3.2

import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message

SYSTEM_PROMPT = """You are a math expert. Solve math problems step by step.
Show your reasoning clearly. Be concise."""


class MathAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()

        # Call the local LLM via Ollama
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )
        answer = response["message"]["content"]
        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(new_agent_text_message(answer))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Math Agent",
    description="Solves math problems using llama3.2 running via Ollama.",
    url="http://localhost:10002/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="math-solver",
            name="Math Solver",
            description="Solve arithmetic, algebra, and basic calculus problems.",
            tags=["math", "calculation", "algebra"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=MathAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10002, log_level="warning")
'''

with open("math_agent.py", "w") as f:
    f.write(math_agent_code.strip())

print("math_agent.py written!")

In [ ]:
# Start the math agent
math_proc = subprocess.Popen(
    ["python", "math_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

if math_proc.poll() is None:
    print("Math agent running (PID:", math_proc.pid, ")")
else:
    stdout, stderr = math_proc.communicate()
    print("Agent failed to start:", stderr.decode())

In [ ]:
# Ask the Math Agent a question
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10002", "What is 17 * 23 + the square root of 144?")
)
print(answer)

In [ ]:
# Try another question
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10002", "If I have 3x + 7 = 22, what is x?")
)
print(answer)

## Step 8 — The Orchestrator Pattern: A2A's killer feature

Now we unlock what makes A2A powerful: **agents calling other agents**.

We will build:
- A **Writer Agent** (port 10003) — handles writing, editing, summarization
- An **Orchestrator Agent** (port 10004) — receives any request, decides who to delegate to,
  calls the specialist via A2A, and returns the result

```
  User
   │
   ▼
  Orchestrator (10004)
   │  uses LLM to classify the request
   ├──► Math Agent (10002)   if it's a math problem
   │         │
   │         ◄── math answer
   │
   └──► Writer Agent (10003) if it's a writing task
             │
             ◄── written text
   │
   ▼
  User gets the result
```

The orchestrator itself is an A2A agent — it can be composed into larger systems.
This is the core of **multi-agent architectures**.

In [ ]:
writer_agent_code = '''
# writer_agent.py — A2A agent for writing and summarization tasks

import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill
from a2a.utils import new_agent_text_message

SYSTEM_PROMPT = """You are a skilled writer and editor.
Help with drafting emails, stories, and essays. Improve tone and clarity.
Summarize long texts into clear, concise bullet points.
Keep responses focused and polished."""


class WriterAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )
        answer = response["message"]["content"]
        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(new_agent_text_message(answer))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Writer Agent",
    description="Drafts, edits, and summarizes text using llama3.2.",
    url="http://localhost:10003/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="write",
            name="Writing Assistant",
            description="Draft, edit, and improve written content.",
            tags=["writing", "editing", "email"],
        ),
        AgentSkill(
            id="summarize",
            name="Summarizer",
            description="Summarize long text into concise bullet points.",
            tags=["summarization", "tldr"],
        ),
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=WriterAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10003, log_level="warning")
'''

with open("writer_agent.py", "w") as f:
    f.write(writer_agent_code.strip())

print("writer_agent.py written!")

In [ ]:
orchestrator_code = '''
# orchestrator_agent.py
# An A2A agent that routes requests to specialist agents via A2A protocol.

import uvicorn
import ollama
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.apps import A2AStarletteApplication
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, Task, Message
from a2a.client import ClientFactory
from a2a.client.helpers import create_text_message_object
from a2a.utils import new_agent_text_message

MATH_AGENT_URL   = "http://localhost:10002"
WRITER_AGENT_URL = "http://localhost:10003"

ROUTER_PROMPT = """You are a routing assistant. Classify the request as exactly one word:
- "math"    if it involves numbers, calculations, equations, or formulas
- "writing" if it involves text, drafting, editing, or summarization
Reply with ONLY the single word."""


def extract_text(event) -> str:
    if isinstance(event, tuple):
        task, _ = event
        for msg in reversed(task.history or []):
            if msg.role.value == "agent":
                for part in msg.parts:
                    if hasattr(part.root, "text"):
                        return part.root.text
    elif isinstance(event, Message):
        for part in event.parts:
            if hasattr(part.root, "text"):
                return part.root.text
    return str(event)


class OrchestratorExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_input = context.get_user_input()

        # ── Step 1: Classify the request with LLM ─────────────────────────────
        classification = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "system", "content": ROUTER_PROMPT},
                {"role": "user",   "content": user_input},
            ],
        )["message"]["content"].strip().lower()

        # ── Step 2: Route to the right specialist ─────────────────────────────
        if "math" in classification:
            target_url  = MATH_AGENT_URL
            target_name = "Math Agent"
        else:
            target_url  = WRITER_AGENT_URL
            target_name = "Writer Agent"

        # ── Step 3: Call the specialist agent via A2A ─────────────────────────
        specialist_client = await ClientFactory.connect(target_url)
        specialist_text = ""
        async for event in specialist_client.send_message(
            create_text_message_object(content=user_input)
        ):
            specialist_text = extract_text(event)
            break

        # ── Step 4: Return the specialist\'s answer to the original caller ────
        reply = f"[Routed to {target_name}]\\n\\n{specialist_text}"
        # enqueue_event is async in a2a-sdk >= 0.3.x — must be awaited
        await event_queue.enqueue_event(new_agent_text_message(reply))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


agent_card = AgentCard(
    name="Orchestrator Agent",
    description="Routes user requests to Math or Writer specialist agents.",
    url="http://localhost:10004/",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="route",
            name="Smart Router",
            description="Automatically delegates tasks to the right specialist agent.",
            tags=["orchestration", "routing"],
        )
    ],
    defaultInputModes=["text/plain"],
    defaultOutputModes=["text/plain"],
)

request_handler = DefaultRequestHandler(
    agent_executor=OrchestratorExecutor(),
    task_store=InMemoryTaskStore(),
)

app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=10004, log_level="warning")
'''

with open("orchestrator_agent.py", "w") as f:
    f.write(orchestrator_code.strip())

print("orchestrator_agent.py updated!")

## Step 9 — Start all agents and test the full pipeline

In [ ]:
# Start Writer Agent and Orchestrator Agent
# (Math Agent is already running from Step 7)

writer_proc = subprocess.Popen(
    ["python", "writer_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
orchestrator_proc = subprocess.Popen(
    ["python", "orchestrator_agent.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)  # wait for both to boot

all_procs = {
    "Math Agent      (10002)": math_proc,
    "Writer Agent    (10003)": writer_proc,
    "Orchestrator    (10004)": orchestrator_proc,
}
for name, proc in all_procs.items():
    status = "running" if proc.poll() is None else "FAILED"
    print(f"  {name}  →  {status}")

In [ ]:
# Test 1: A math question — should be routed to Math Agent
print("=== Math question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent("http://localhost:10004", "What is 256 divided by 16, minus 4 squared?")
)
print(answer)

In [ ]:
# Test 2: A writing question — should be routed to Writer Agent
print("=== Writing question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent(
        "http://localhost:10004",
        "Write a short professional email declining a meeting invitation politely."
    )
)
print(answer)

In [ ]:
# Test 3: Summarization — should be routed to Writer Agent
print("=== Summarization question ===")
answer = asyncio.get_event_loop().run_until_complete(
    call_agent(
        "http://localhost:10004",
        "Summarize what A2A protocol is in 3 bullet points."
    )
)
print(answer)

In [ ]:
# Clean up — stop all agent processes
for name, proc in all_procs.items():
    proc.terminate()
    proc.wait()
    print(f"Stopped {name.strip()}")

## Step 10 — A2A vs MCP: when to use which

| | MCP | A2A |
|---|---|---|
| **Transport** | stdio (or HTTP) | HTTP (always) |
| **Discovery** | Server spawned as subprocess | Agent Card at `/.well-known/agent.json` |
| **Unit of work** | Tool call (function + args) | Task (message + history) |
| **Who calls** | LLM decides which tool to call | Agent decides which agent to delegate to |
| **State** | Stateless by default | Tasks have lifecycle (submitted → completed) |
| **Best for** | LLM augmented with capabilities | Agents collaborating on complex tasks |
| **Examples** | DB query, file read, web search | Research + write + review pipeline |

### They are complementary

A real production system often uses **both**:
- Each A2A agent internally uses MCP tools to access data
- An A2A orchestrator coordinates multiple specialist agents

```
  User
   │  A2A
   ▼
  Orchestrator Agent
   │  A2A                  │  A2A
   ▼                       ▼
  Research Agent        Writer Agent
   │  MCP                  │  MCP
   ▼                       ▼
  web_search()          grammar_check()
  fetch_doc()           style_guide()
```

---

## Summary — What you built

| Step | What you did |
|---|---|
| 1 | Installed `a2a-sdk`, `uvicorn`, `httpx` |
| 2 | Built and inspected an `AgentCard` |
| 3 | Wrote an `AgentExecutor` subclass (echo agent) |
| 4 | Started the agent as an HTTP service and fetched its card |
| 5 | Used `A2AClient` to send messages and parsed the JSON-RPC response |
| 6 | Understood the Task lifecycle: submitted → working → completed |
| 7 | Built an Ollama-backed Math Agent |
| 8 | Built a Writer Agent and an Orchestrator that routes between them |
| 9 | Tested the full multi-agent pipeline end-to-end |
| 10 | Compared A2A with MCP and learned when to use each |

### Where to go next

- **Streaming responses**: set `capabilities=AgentCapabilities(streaming=True)` and use `client.send_message_streaming()`
- **Push notifications**: agents can call back when long tasks complete
- **Auth**: add `securitySchemes` to the AgentCard for Bearer token auth
- **Persistent task store**: replace `InMemoryTaskStore` with a Redis or database backend
- **Agent registry**: build a service that indexes Agent Cards for discovery
- **A2A + MCP together**: let each A2A agent use MCP tools internally
- **Official samples**: https://github.com/a2aproject/a2a-samples
- **A2A protocol docs**: https://a2aprotocol.ai